<a href="https://colab.research.google.com/github/treasureacademyageva/treasureacademyageva/blob/preview/model/notebooks/tiny_100_step_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treasure EDU — exact 100-step GPU benchmark

This notebook uses a free Google Colab GPU when one is available. Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. It benchmarks only 100 optimiser steps and never starts a long training run.

In [1]:
import os, shutil, subprocess, sys
work = '/content/treasureacademyageva'
shutil.rmtree(work, ignore_errors=True)
subprocess.run(['git','clone','--depth','1','--branch','preview','https://github.com/treasureacademyageva/treasureacademyageva.git',work], check=True)
os.chdir(work + '/model')
subprocess.run([sys.executable,'-m','pip','install','-q','tokenizers>=0.20,<1','pypdf>=6,<7'], check=True)
print('Repository and lightweight dependencies are ready.')

Repository and lightweight dependencies are ready.


In [2]:
import torch
assert torch.cuda.is_available(), 'No GPU is attached. Choose Runtime > Change runtime type > T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
subprocess.run(['nvidia-smi'], check=True)

GPU: Tesla T4
VRAM GiB: 14.56


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [3]:
commands = [
 ['python3','scripts/build_curriculum.py'],
 ['python3','scripts/fetch_open_data.py','--strict'],
 ['python3','scripts/validate_sources.py'],
 ['python3','scripts/prepare_data.py','--config','config/micro.json'],
 ['python3','tokenizer/train_tokenizer.py','--config','config/tiny.json'],
 ['python3','scripts/prepare_data.py','--config','config/tiny.json'],
 ['python3','scripts/audit_corpus.py']
]
for command in commands:
 print('\n$', ' '.join(command), flush=True)
 subprocess.run(command, check=True)


$ python3 scripts/build_curriculum.py

$ python3 scripts/fetch_open_data.py --strict

$ python3 scripts/validate_sources.py

$ python3 scripts/prepare_data.py --config config/micro.json

$ python3 tokenizer/train_tokenizer.py --config config/tiny.json

$ python3 scripts/prepare_data.py --config config/tiny.json

$ python3 scripts/audit_corpus.py


In [4]:
report = 'runs/tiny-gpu-benchmark.json'
command = [
 'python3','scripts/benchmark_tiny.py','--steps','100',
 '--provider','Google Colab Free','--instance','Colab assigned CUDA GPU',
 '--hourly-cost-usd','0','--target-steps','50000','--output',report
]
subprocess.run(command, check=True)
print('Exact 100-step benchmark completed. No long run was started.')

Exact 100-step benchmark completed. No long run was started.


In [5]:
import json
from google.colab import files
result = json.load(open(report))
assert result['kind'] == 'GPU' and result['benchmark']['steps'] == 100
print(json.dumps({
 'gpu': result['gpu'],
 'benchmark': result['benchmark'],
 'projection': result['projection'],
 'sample': result['sample'],
 'long_run_authorised': result['long_run_authorised']
}, indent=2))
files.download(report)

{
  "gpu": {
    "name": "Tesla T4",
    "vram_gb": 14.56,
    "compute_capability": "7.5"
  },
  "benchmark": {
    "steps": 100,
    "warmup_steps_excluded": 5,
    "batch_size": 16,
    "block_size": 512,
    "gradient_accumulation": 8,
    "tokens_per_step": 65536,
    "seconds_per_step": 3.053895085684218,
    "tokens_per_second": 21459.80728257952,
    "initial_training_loss": 9.053499221801758,
    "final_training_loss": 4.881256401538849,
    "mean_first_10_training_loss": 7.83900437951088,
    "mean_final_10_training_loss": 4.6828012585639955,
    "initial_validation_loss": 9.045355606079102,
    "final_validation_loss": 4.789897727966308,
    "peak_allocated_vram_gb": 2.999,
    "peak_reserved_vram_gb": 3.166
  },
  "projection": {
    "target_steps": 50000,
    "estimated_hours": 42.41520952339192,
    "hourly_cost_usd": 0.0,
    "estimated_cost_usd": 0.0
  },
  "sample": "A good learner asks of the same.\n\n\n\n\n\nThere are not the Queen will show it\n\n\n\nThe soil. At th

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>